# M-IRT Dimensionality Experiment

**Goal**: Determine the intrinsic dimensionality of student proficiency in the PIX dataset by fitting Multidimensional IRT models (M-IRT) with increasing latent dimensions K = 1, 2, 4, 8, 16.

**M-IRT model**:  
$$P(\text{correct} \mid \theta_u, a_j, d_j) = \sigma(a_j \cdot \theta_u - d_j)$$

- $\theta_u \in \mathbb{R}^K$: student proficiency vector  
- $a_j \in \mathbb{R}^K_{+}$: item discrimination vector (constrained positive for identifiability)  
- $d_j \in \mathbb{R}$: item difficulty scalar  

**Evaluation**: held-out log-likelihood and accuracy on a 20% user-based test split.

**Secondary analysis**: do learned $\theta$ dimensions align with the 16 competence codes?

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from scipy.stats import spearmanr
from scipy.spatial.distance import pdist, squareform
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cpu


## 1. Data loading and preprocessing

In [2]:
df_raw = pd.read_csv('../data/pix_data.csv', index_col=0)
print(f'Raw shape: {df_raw.shape}')
print(f'Competences: {sorted(df_raw["competence_code"].unique())}')

Raw shape: (3222679, 6)
Competences: [np.float64(1.1), np.float64(1.2), np.float64(1.3), np.float64(2.1), np.float64(2.2), np.float64(2.3), np.float64(2.4), np.float64(3.1), np.float64(3.2), np.float64(3.3), np.float64(3.4), np.float64(4.1), np.float64(4.2), np.float64(4.3), np.float64(5.1), np.float64(5.2)]


In [3]:
# Keep only clear binary outcomes; drop abandoned / timed-out responses
df = df_raw[df_raw['answer_result'].isin(['ok', 'ko'])].copy()
df['correct'] = (df['answer_result'] == 'ok').astype(np.int8)

# Filter challenges with fewer than MIN_OBS observations (too sparse for stable estimates)
MIN_OBS = 50
challenge_counts = df['challenge_id'].value_counts()
valid_challenges = challenge_counts[challenge_counts >= MIN_OBS].index
df = df[df['challenge_id'].isin(valid_challenges)].copy()

# Filter users with fewer than MIN_USER_OBS responses after challenge filtering
MIN_USER_OBS = 5
user_counts = df['user_id'].value_counts()
valid_users = user_counts[user_counts >= MIN_USER_OBS].index
df = df[df['user_id'].isin(valid_users)].copy()

print(f'After filtering: {df.shape[0]:,} responses')
print(f'  Users:      {df["user_id"].nunique():,}')
print(f'  Challenges: {df["challenge_id"].nunique():,}')
print(f'  Competences:{df["competence_code"].nunique()}')

# Integer encode user and challenge IDs
user_enc = LabelEncoder().fit(df['user_id'])
challenge_enc = LabelEncoder().fit(df['challenge_id'])

df['uid'] = user_enc.transform(df['user_id'])
df['cid'] = challenge_enc.transform(df['challenge_id'])

N_USERS = df['uid'].nunique()
N_CHALLENGES = df['cid'].nunique()
print(f'\nN_USERS={N_USERS}, N_CHALLENGES={N_CHALLENGES}')

After filtering: 2,414,940 responses
  Users:      99,910
  Challenges: 592
  Competences:16

N_USERS=99910, N_CHALLENGES=592


In [4]:
# Build challenge metadata: competence code per challenge (for later analysis)
challenge_meta = (
    df.drop_duplicates('cid')[['cid', 'challenge_id', 'skill_id', 'competence_code']]
    .set_index('cid')
    .sort_index()
)
print('Challenge meta sample:')
challenge_meta.head()

Challenge meta sample:


,challenge_id,skill_id,competence_code
cid,,,
0,challenge107G8qLjbYqYH9,skill2J10PC9aA2xaXy,4.1
1,challenge11rcXvP06PFyxr,skillPuDp1iFG72CuB,2.1
2,challenge11tkvfyyzcmoLQ,recnLN4ZCdZdTC32I,1.1
3,challenge11wR6fD4z3sdUr,skill28dr9V7BawgagX,2.2
4,challenge12Au3JWnhVOBJf,skill172eGR3lHlFdeM,2.3


In [5]:
# Train / test split: hold out 20% of USERS entirely
all_users = np.arange(N_USERS)
train_users, test_users = train_test_split(all_users, test_size=0.2, random_state=42)
train_users_set = set(train_users)

df_train = df[df['uid'].isin(train_users_set)].copy()
df_test  = df[~df['uid'].isin(train_users_set)].copy()

print(f'Train: {len(df_train):,} responses, {df_train["uid"].nunique():,} users')
print(f'Test:  {len(df_test):,} responses,  {df_test["uid"].nunique():,} users')

def to_tensors(df_split):
    return (
        torch.tensor(df_split['uid'].values,     dtype=torch.long),
        torch.tensor(df_split['cid'].values,     dtype=torch.long),
        torch.tensor(df_split['correct'].values, dtype=torch.float32),
    )

tr_u, tr_c, tr_y = to_tensors(df_train)
te_u, te_c, te_y = to_tensors(df_test)

Train: 1,931,453 responses, 79,928 users
Test:  483,487 responses,  19,982 users


## 2. M-IRT model definition

In [6]:
class MIRT(nn.Module):
    """
    Multidimensional 2PL IRT:
        logit = softplus(a_j) · theta_u - d_j
    
    Using softplus on discrimination to keep a_j > 0 (identifiability).
    For K=1 this reduces to standard 2PL IRT.
    """
    def __init__(self, n_users: int, n_challenges: int, K: int):
        super().__init__()
        self.K = K
        # Student proficiency: N x K, init near 0
        self.theta = nn.Embedding(n_users, K)
        nn.init.normal_(self.theta.weight, 0, 0.1)
        
        # Item discrimination: M x K, init near 1 (before softplus)
        self.a_raw = nn.Embedding(n_challenges, K)
        nn.init.constant_(self.a_raw.weight, 0.5)   # softplus(0.5) ≈ 0.97
        
        # Item difficulty: M x 1
        self.d = nn.Embedding(n_challenges, 1)
        nn.init.zeros_(self.d.weight)

    def forward(self, u_idx, c_idx):
        theta = self.theta(u_idx)                          # (B, K)
        a = torch.nn.functional.softplus(self.a_raw(c_idx))  # (B, K), positive
        d = self.d(c_idx).squeeze(-1)                       # (B,)
        logit = (a * theta).sum(dim=-1) - d                # (B,)
        return logit

    def predict_proba(self, u_idx, c_idx):
        return torch.sigmoid(self.forward(u_idx, c_idx))

print('MIRT class defined.')

MIRT class defined.


## 3. Training loop

In [7]:
def train_mirt(
    K: int,
    tr_u, tr_c, tr_y,
    te_u, te_c, te_y,
    n_users: int,
    n_challenges: int,
    n_epochs: int = 30,
    batch_size: int = 4096,
    lr: float = 5e-3,
    l2: float = 1e-3,
    verbose: bool = True,
):
    model = MIRT(n_users, n_challenges, K).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=l2)
    criterion = nn.BCEWithLogitsLoss()

    dataset = TensorDataset(tr_u, tr_c, tr_y)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    history = {'train_loss': [], 'test_loss': [], 'test_acc': []}

    for epoch in range(1, n_epochs + 1):
        model.train()
        total_loss, n_batches = 0.0, 0
        for u_b, c_b, y_b in loader:
            u_b, c_b, y_b = u_b.to(DEVICE), c_b.to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad()
            logits = model(u_b, c_b)
            loss = criterion(logits, y_b)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
        train_loss = total_loss / n_batches

        # Eval on test set (no grad, chunked to avoid OOM)
        model.eval()
        with torch.no_grad():
            chunk = 8192
            logits_all, y_all = [], []
            for i in range(0, len(te_u), chunk):
                u_b = te_u[i:i+chunk].to(DEVICE)
                c_b = te_c[i:i+chunk].to(DEVICE)
                y_b = te_y[i:i+chunk]
                logits_all.append(model(u_b, c_b).cpu())
                y_all.append(y_b)
            logits_all = torch.cat(logits_all)
            y_all      = torch.cat(y_all)
            test_loss  = criterion(logits_all, y_all).item()
            preds      = (logits_all > 0).float()
            test_acc   = (preds == y_all).float().mean().item()

        history['train_loss'].append(train_loss)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)

        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f'  K={K} | Epoch {epoch:3d} | '
                  f'train_loss={train_loss:.4f} | '
                  f'test_loss={test_loss:.4f} | '
                  f'test_acc={test_acc:.4f}')

    return model, history

print('Training function defined.')

Training function defined.


## 4. Sweep over K

In [8]:
DIMS = [1, 2, 4, 8, 16]
N_EPOCHS = 50

results = {}   # K -> {model, history}

for K in DIMS:
    print(f'\n=== Training M-IRT with K={K} ===')
    model, history = train_mirt(
        K=K,
        tr_u=tr_u, tr_c=tr_c, tr_y=tr_y,
        te_u=te_u, te_c=te_c, te_y=te_y,
        n_users=N_USERS,
        n_challenges=N_CHALLENGES,
        n_epochs=N_EPOCHS,
        batch_size=8192,
        lr=5e-3,
        l2=1e-3,
        verbose=True,
    )
    results[K] = {'model': model, 'history': history}

print('\nDone.')


=== Training M-IRT with K=1 ===
  K=1 | Epoch   1 | train_loss=0.5879 | test_loss=0.5278 | test_acc=0.8035
  K=1 | Epoch   5 | train_loss=0.4755 | test_loss=0.4754 | test_acc=0.8034
  K=1 | Epoch  10 | train_loss=0.4740 | test_loss=0.4746 | test_acc=0.8035
  K=1 | Epoch  15 | train_loss=0.4740 | test_loss=0.4746 | test_acc=0.8033
  K=1 | Epoch  20 | train_loss=0.4740 | test_loss=0.4745 | test_acc=0.8036
  K=1 | Epoch  25 | train_loss=0.4740 | test_loss=0.4746 | test_acc=0.8035
  K=1 | Epoch  30 | train_loss=0.4740 | test_loss=0.4744 | test_acc=0.8035
  K=1 | Epoch  35 | train_loss=0.4740 | test_loss=0.4746 | test_acc=0.8034
  K=1 | Epoch  40 | train_loss=0.4740 | test_loss=0.4747 | test_acc=0.8030
  K=1 | Epoch  45 | train_loss=0.4740 | test_loss=0.4746 | test_acc=0.8033
  K=1 | Epoch  50 | train_loss=0.4740 | test_loss=0.4747 | test_acc=0.8034

=== Training M-IRT with K=2 ===
  K=2 | Epoch   1 | train_loss=0.5878 | test_loss=0.5278 | test_acc=0.8037
  K=2 | Epoch   5 | train_loss=0.4

## 5. Learning curves and summary metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = cm.viridis(np.linspace(0, 1, len(DIMS)))

for ax, metric, ylabel in zip(axes,
                               ['test_loss', 'test_acc'],
                               ['Test BCE loss (↓)', 'Test accuracy (↑)']):
    for K, color in zip(DIMS, colors):
        h = results[K]['history']
        ax.plot(h[metric], label=f'K={K}', color=color)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('M-IRT: learning curves by dimension K', fontsize=14)
plt.tight_layout()
plt.savefig('../results/mirt_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/mirt_learning_curves.png')

In [ ]:
# Summary table
rows = []
for K in DIMS:
    h = results[K]['history']
    model = results[K]['model']
    n_params = sum(p.numel() for p in model.parameters())
    rows.append({
        'K': K,
        'n_params': n_params,
        'best_test_loss': min(h['test_loss']),
        'final_test_loss': h['test_loss'][-1],
        'best_test_acc':  max(h['test_acc']),
        'final_test_acc': h['test_acc'][-1],
    })

summary = pd.DataFrame(rows).set_index('K')
summary['delta_loss_vs_K1'] = summary['final_test_loss'] - summary.loc[1, 'final_test_loss']
summary['delta_acc_vs_K1']  = summary['final_test_acc']  - summary.loc[1, 'final_test_acc']
print(summary.to_string(float_format='%.5f'))

## 6. Do latent dimensions align with competences? (K=16 analysis)

In [ ]:
# Extract user theta for K=16 model
K_ANALYZE = 16
model16 = results[K_ANALYZE]['model']
model16.eval()

with torch.no_grad():
    theta_train = model16.theta(torch.tensor(train_users, dtype=torch.long).to(DEVICE)).cpu().numpy()

print(f'theta shape (train users): {theta_train.shape}')

# For each train user, compute their correct rate per competence
df_train_analysis = df_train.copy()
# Map uid to row index in theta_train
train_uid_to_row = {uid: i for i, uid in enumerate(train_users)}

competences = sorted(df['competence_code'].unique())

# Per-user correct rate per competence
user_comp_correct = (
    df_train_analysis.groupby(['uid', 'competence_code'])['correct'].mean()
    .unstack('competence_code')
    .reindex(index=train_users)  # align to theta rows
)
print(f'User x competence correct rate matrix: {user_comp_correct.shape}')
print(f'Sparsity (NaN): {user_comp_correct.isna().mean().mean():.1%}')

In [ ]:
# Spearman correlation: each theta dim vs each competence correct rate
# Only use users who have responses for a given competence
corr_matrix = np.zeros((K_ANALYZE, len(competences)))
pval_matrix = np.zeros((K_ANALYZE, len(competences)))

for d_idx in range(K_ANALYZE):
    for c_idx, comp in enumerate(competences):
        col = user_comp_correct[comp]
        valid = col.notna().values
        if valid.sum() < 10:
            corr_matrix[d_idx, c_idx] = np.nan
            continue
        r, p = spearmanr(theta_train[valid, d_idx], col.values[valid])
        corr_matrix[d_idx, c_idx] = r
        pval_matrix[d_idx, c_idx] = p

corr_df = pd.DataFrame(
    corr_matrix,
    index=[f'dim_{i}' for i in range(K_ANALYZE)],
    columns=[f'comp_{c}' for c in competences]
)

print('Spearman correlations (theta dim vs competence correct rate):')
print(corr_df.round(3).to_string())

In [ ]:
import matplotlib.ticker as ticker

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(corr_matrix, cmap='RdBu', vmin=-0.6, vmax=0.6, aspect='auto')
plt.colorbar(im, ax=ax, label='Spearman ρ')
ax.set_xticks(range(len(competences)))
ax.set_xticklabels([str(c) for c in competences], rotation=45, ha='right')
ax.set_yticks(range(K_ANALYZE))
ax.set_yticklabels([f'dim {i}' for i in range(K_ANALYZE)])
ax.set_xlabel('Competence code')
ax.set_ylabel('Latent dimension')
ax.set_title(f'M-IRT (K={K_ANALYZE}): correlation of latent dims with per-competence correct rate')
plt.tight_layout()
plt.savefig('../results/mirt_dim_competence_corr.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Which dimension best predicts each competence?
best_dim_per_comp = np.nanargmax(np.abs(corr_matrix), axis=0)
best_corr_per_comp = corr_matrix[best_dim_per_comp, np.arange(len(competences))]

comp_dim_summary = pd.DataFrame({
    'competence': competences,
    'best_dim': best_dim_per_comp,
    'best_spearman_rho': best_corr_per_comp,
}).set_index('competence')

print('Best latent dimension per competence:')
print(comp_dim_summary.to_string(float_format='%.3f'))

## 7. Discrimination vectors: do items from the same competence cluster in latent space?

In [ ]:
with torch.no_grad():
    a_raw = model16.a_raw.weight.cpu().numpy()  # (N_CHALLENGES, K)
    a_pos = np.log1p(np.exp(a_raw))             # softplus = log(1+exp(x))

# Normalize each row (unit discrimination vector)
a_norm = a_pos / (np.linalg.norm(a_pos, axis=1, keepdims=True) + 1e-8)

# Assign competence to each challenge
challenge_comp = challenge_meta['competence_code'].values  # indexed by cid

# PCA of discrimination vectors colored by competence
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=42)
a_2d = pca.fit_transform(a_norm)

comp_codes = sorted(df['competence_code'].unique())
comp_to_int = {c: i for i, c in enumerate(comp_codes)}
colors_per_challenge = [comp_to_int[c] for c in challenge_comp]

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    a_2d[:, 0], a_2d[:, 1],
    c=colors_per_challenge, cmap='tab20', s=20, alpha=0.7
)
cbar = plt.colorbar(scatter, ax=ax, ticks=range(len(comp_codes)))
cbar.set_ticklabels([str(c) for c in comp_codes])
cbar.set_label('Competence code')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
ax.set_title(f'PCA of item discrimination vectors (K={K_ANALYZE}) — colored by competence')
plt.tight_layout()
plt.savefig('../results/mirt_discrimination_pca.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Silhouette score: how well do discrimination vectors cluster by competence?
from sklearn.metrics import silhouette_score

labels = np.array(colors_per_challenge)
# Only compute if we have enough samples
if len(np.unique(labels)) > 1 and len(labels) > 10:
    sil = silhouette_score(a_norm, labels, metric='cosine')
    print(f'Silhouette score (cosine, by competence): {sil:.4f}')
    print('  (> 0 means competences are partially separable in discrimination space)')

# Also report per-competence centroid similarity
centroids = {}
for comp in comp_codes:
    mask = challenge_comp == comp
    if mask.sum() > 0:
        c = a_norm[mask].mean(axis=0)
        centroids[comp] = c / (np.linalg.norm(c) + 1e-8)

centroid_matrix = np.stack([centroids[c] for c in comp_codes])
sim_matrix = centroid_matrix @ centroid_matrix.T

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Cosine similarity')
labels_str = [str(c) for c in comp_codes]
ax.set_xticks(range(len(comp_codes)))
ax.set_yticks(range(len(comp_codes)))
ax.set_xticklabels(labels_str, rotation=45, ha='right')
ax.set_yticklabels(labels_str)
ax.set_title('Cosine similarity of per-competence discrimination centroids')
plt.tight_layout()
plt.savefig('../results/mirt_competence_centroid_sim.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Elbow plot: is there a natural K?

In [ ]:
final_losses = {K: results[K]['history']['test_loss'][-1] for K in DIMS}
final_accs   = {K: results[K]['history']['test_acc'][-1]  for K in DIMS}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(DIMS, [final_losses[K] for K in DIMS], 'o-', color='steelblue', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of latent dimensions K')
axes[0].set_ylabel('Test BCE loss')
axes[0].set_title('Elbow: test loss vs K')
axes[0].set_xticks(DIMS)
axes[0].grid(True, alpha=0.3)

axes[1].plot(DIMS, [final_accs[K] for K in DIMS], 'o-', color='tomato', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of latent dimensions K')
axes[1].set_ylabel('Test accuracy')
axes[1].set_title('Elbow: test accuracy vs K')
axes[1].set_xticks(DIMS)
axes[1].grid(True, alpha=0.3)

plt.suptitle('M-IRT: performance vs. number of latent dimensions', fontsize=14)
plt.tight_layout()
plt.savefig('../results/mirt_elbow.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nFinal metrics:')
for K in DIMS:
    print(f'  K={K:2d}  loss={final_losses[K]:.5f}  acc={final_accs[K]:.5f}')

## 9. Summary

- **Accuracy vs K**: check the elbow plot — if accuracy plateaus after K=2 or K=4, a low-dimensional model is sufficient.
- **Dimension-competence correlation**: if particular latent dimensions strongly correlate with specific competence correct rates, it suggests proficiency is partly structured by competence domain.
- **Discrimination clustering**: silhouette score and centroid similarity reveal whether items from the same competence "point in the same direction" in latent space — i.e., whether the model naturally groups skills.

**Next experiments to consider**:
1. **Confirmatory M-IRT**: fix item loadings based on the competence structure (block-diagonal discrimination matrix) and compare to exploratory M-IRT.
2. **Hierarchical IRT**: model the competence → skill → challenge hierarchy explicitly.
3. **NMF / SVD on the response matrix**: compare to IRT dimensionality estimates.
4. **Cross-competence transfer**: does high proficiency on competence X predict performance on Y? (inter-competence correlation from theta).